<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 4. Аутентификация и авторизация — подробная методичка

Эта методичка продолжает Этап 3. Предполагается, что у вас уже есть:

- работающее приложение с `/health` и `/healthz`;
- модели SQLAlchemy с таблицей `users`;
- Docker Compose с `db` и `app`;
- зелёные тесты: `test_config.py`, `test_health.py`, `test_models.py`;
- 25+ пройденных тестов.

Если чего-то нет — вернитесь к Этапу 3.

**Цель этапа:** регистрация, вход и система ролей через JWT.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `backend/utils/security.py` — хеширование паролей (bcrypt) и работа с JWT-токенами;
- `backend/api/deps.py` — зависимости `get_current_user`, `require_admin`, `require_editor`, `optional_user`;
- `backend/api/routes/auth.py` — роуты `/auth/register`, `/auth/login`, `/auth/me`;
- обновлённый `backend/api/main.py` — подключение роутера `auth` (только `auth` и `health` на этом этапе);
- `tests/unit/test_security.py` — тесты на bcrypt и JWT;
- `tests/integration/test_auth.py` — полный цикл регистрации/логина/`/me`;
- понимание, как работает `psql`-CRUD для таблицы `users`.

**Критерии готовности:**

- [ ] Пароли в БД хранятся только в виде bcrypt-хеша (`$2b$12$...`, длина 60).
- [ ] `POST /auth/register` создаёт пользователя и возвращает JWT.
- [ ] `POST /auth/login` возвращает токен при верных данных, 401 при неверных.
- [ ] `GET /auth/me` без токена → 401, с токеном → данные пользователя.
- [ ] Токен живёт `JWT_EXPIRATION_MINUTES` минут.
- [ ] `python -m poetry run pytest tests/ -v` проходит без реальной Postgres.
- [ ] Ручной цикл register → login → /me работает через `curl`.

**Важное предупреждение:** на этом этапе подключаются **только роутеры `auth` и `health`**. Роутеры `translate`, `admin`, `eval`, `hitl` появятся на этапах 9–10. Не пытайтесь подключить их раньше — приложение не запустится с `ModuleNotFoundError`. Если вы видели финальный репозиторий проекта, не пугайтесь: `main.py` там выглядит иначе.

---

## 4.0. Подготовка: ветка в Git

Откройте PowerShell, перейдите в папку проекта:

```powershell
cd D:\RUNG
```

Убедитесь, что ветка `main` чистая:

```powershell
git status
```

**Что вы должны увидеть:**

```
On branch main
nothing to commit, working tree clean
```

Создайте ветку:

```powershell
git checkout -b week-4
```

**Что вы должны увидеть:**

```
Switched to a new branch 'week-4'
```

---

## 4.1. Проверка `JWT_SECRET_KEY`

**Почему это первым шагом:** если `JWT_SECRET_KEY` слабый или дефолтный, любой сможет подделать токен и войти под чужим аккаунтом. Это критично.

**Проверьте `.env` (в корне проекта):**

```powershell
type .env | Select-String "JWT_SECRET_KEY"
```

**Если увидели `change-me-in-production` или `change_me_to_a_long_random_string`** — сгенерируйте нормальный ключ:

```powershell
python -c "import secrets; print(secrets.token_urlsafe(64))"
```

**Что вы должны увидеть:** длинную случайную строку (~86 символов):

```
hZ3Q1r9XmLkP2vN8sB4tY7wC5dF6gH0jK1aE3bR9cM2nP4vQ8sW1xY5zA7cD0eF3gH6iJ9kL2mN
```

Скопируйте её и вставьте в **оба** файла:

- `.env` (корень проекта);
- `infrastructure/.env` (Docker).

**Проверьте, что ключ не попал в Git:**

```powershell
git check-ignore -v .env
git check-ignore -v infrastructure/.env
```

**Что вы должны увидеть:** для обоих файлов строку из `.gitignore`. Если хоть один не игнорируется — добавьте в `.gitignore`:

```
.env
.env.*
!.env.example
```

**Проверьте, что в истории Git нет реального ключа:**

```powershell
git log --all --full-history -- .env
```

**Что вы должны увидеть:** пусто или только `.env.example`. Если есть настоящий `.env` — **немедленно смените `JWT_SECRET_KEY`** (он уже в истории).

---

## 4.2. `backend/utils/security.py` — хеширование и JWT

**Что делает модуль:**

1. Хеширует пароли через bcrypt и проверяет их.
2. Создаёт и декодирует JWT-токены.

**Что такое bcrypt:** алгоритм хеширования паролей. Отличается от MD5/SHA256 тем, что **медленный** — это его преимущество. Злоумышленник не сможет быстро подобрать пароль перебором, даже если украдёт базу. Плюс bcrypt добавляет «соль» (случайные данные) к каждому хешу, поэтому одинаковые пароли дают разные хеши.

**Что такое JWT:** JSON Web Token — компактный способ передать подписанные данные между клиентом и сервером. Состоит из трёх частей (header.payload.signature), закодированных Base64URL и разделённых точками. Сервер подписывает токен секретным ключом — клиент не может его подделать. Мы кладём в токен `sub` (id пользователя) и `role`. При каждом запросе получаем их обратно без обращения к БД.

### Создание файла

Создайте `backend/utils/security.py`:

```python
# backend/utils/security.py
"""
Хеширование паролей (bcrypt) и работа с JWT-токенами.

Пароли НИКОГДА не хранятся в открытом виде. В базе лежит bcrypt-хеш.
При логине мы хешируем введённый пароль и сравниваем с хешем из БД.
"""

import logging
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, Optional

from fastapi import HTTPException
from jose import JWTError, jwt
from passlib.context import CryptContext

from backend.core.config import get_settings

logger = logging.getLogger(__name__)
settings = get_settings()


# ============================================================================
# Контекст для хеширования паролей
# ============================================================================
# schemes=["bcrypt"] — использовать bcrypt.
# deprecated="auto" — автоматически помечать устаревшие хеши,
# если в будущем поменяем схему.
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")


# ============================================================================
# Константы JWT
# ============================================================================
SECRET_KEY = settings.JWT_SECRET_KEY
ALGORITHM = settings.JWT_ALGORITHM
ACCESS_TOKEN_EXPIRE_MINUTES = settings.JWT_EXPIRATION_MINUTES


# ============================================================================
# Пароли
# ============================================================================
def verify_password(plain_password: str, hashed_password: str) -> bool:
    """
    Проверяет, что открытый пароль соответствует хешу.

    Возвращает True/False. Исключений не бросает.
    """
    return pwd_context.verify(plain_password, hashed_password)


def get_password_hash(password: str) -> str:
    """
    Возвращает bcrypt-хеш пароля.

    Каждый вызов даёт разный результат даже для одного и того же пароля —
    потому что bcrypt добавляет случайную соль.
    """
    return pwd_context.hash(password)


# ============================================================================
# JWT-токены
# ============================================================================
def create_access_token(
    data: Dict[str, Any],
    expires_delta: Optional[timedelta] = None,
) -> str:
    """
    Создаёт JWT-токен.

    :param data: полезная нагрузка (например, {"sub": user_id, "role": "user"})
    :param expires_delta: время жизни. Если None — берётся JWT_EXPIRATION_MINUTES.
    :return: строка токена (header.payload.signature)
    """
    to_encode: Dict[str, Any] = {}

    # Приводим все значения к примитивам, которые понимает JWT.
    # jose умеет сериализовать только str/int/float/bool, остальное надо строкой.
    for key, value in data.items():
        if callable(value):
            try:
                to_encode[key] = value()
            except Exception:
                to_encode[key] = str(value)
        elif isinstance(value, (str, int, float, bool)):
            to_encode[key] = value
        else:
            to_encode[key] = str(value)

    if expires_delta:
        expire = datetime.now(timezone.utc) + expires_delta
    else:
        expire = datetime.now(timezone.utc) + timedelta(
            minutes=ACCESS_TOKEN_EXPIRE_MINUTES
        )
    to_encode["exp"] = expire

    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    logger.debug("JWT token created for sub=%s", to_encode.get("sub"))
    return encoded_jwt


def decode_token(token: str) -> Dict[str, Any]:
    """
    Декодирует и проверяет JWT.

    Если токен истёк или подпись не совпадает — бросает HTTPException(401).
    """
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload
    except jwt.ExpiredSignatureError:
        logger.warning("JWT token expired")
        raise HTTPException(status_code=401, detail="Token expired")
    except JWTError as e:
        logger.warning("JWT decode error: %s", e)
        raise HTTPException(status_code=401, detail="Invalid token")
```

### Разбор ключевых решений

**Почему `pwd_context` на уровне модуля:** это объект, который держит внутри кэш соли и других служебных данных. Создавать его на каждый вызов — медленно и бессмысленно.

**Почему `verify_password` не бросает исключений:** bcrypt сравнивает хеши за константное время. Если пароль неверный — вернётся `False`. Это правильно: мы не даём злоумышленнику понять, была ли ошибка «пользователь не найден» или «пароль не тот» (в обоих случаях `/auth/login` вернёт 401 с одинаковым текстом).

**Почему в токене `sub` и `role`:** `sub` (subject) — id пользователя. По нему в `get_current_user` мы находим юзера в БД. `role` — чтобы не делать лишний запрос к БД на каждый защищённый роут.

**Почему `expires_delta` опциональный:** в тестах удобно передать `timedelta(seconds=1)`, чтобы проверить истечение токена. В проде используется дефолт из настроек.

**Почему `datetime.now(timezone.utc)`, а не `datetime.utcnow()`:** `utcnow()` возвращает naive datetime (без таймзоны), и это deprecated в Python 3.12+. `now(timezone.utc)` — aware datetime, корректно работает с библиотекой `jose`.

**Почему в цикле `for key, value in data.items()`:** `jose.jwt.encode` может упасть с `TypeError`, если в payload попадут не-примитивы (например, `datetime`). Наш цикл приводит всё к примитивам. `callable(value)` — на случай, если кто-то передаст функцию.

**Почему `raise HTTPException` в `decode_token`, а не свой `RUNGException`:** это метод на границе HTTP — клиент должен получить именно 401, а не 500. `HTTPException` FastAPI автоматически превращает в JSON-ответ с нужным кодом.

### Про ловушку `passlib` + `bcrypt>=5.0`

**Что за проблема:** в `passlib` версии 1.7.4 есть код, который при инициализации обращается к `bcrypt.__about__.__version__`. В `bcrypt` версии 4.x этот атрибут был. В версии 5.0 его **удалили**. При импорте `passlib` с новым `bcrypt` вылетает `AttributeError` или лог-предупреждение.

**Как мы это решаем:** в `pyproject.toml` у нас:

```toml
bcrypt = "<5.0.0"
passlib = {extras = ["bcrypt"], version = "^1.7.4"}
```

Мы явно запрещаем bcrypt 5.0 и выше.

**Проверка:**

```powershell
python -m poetry run python -c "import bcrypt; print(bcrypt.__version__)"
```

**Что вы должны увидеть:** `4.x.x` (например, `4.2.0`).

**Если увидели `5.x`** — пересоберите lock-файл:

```powershell
python -m poetry lock
python -m poetry install
```

**Проверка, что bcrypt работает:**

```powershell
python -m poetry run python -c "from backend.utils.security import get_password_hash, verify_password; h = get_password_hash('TestPass123'); print(h[:20]); print(verify_password('TestPass123', h)); print(verify_password('WrongPass', h))"
```

**Что вы должны увидеть:**

```
$2b$12$xxxxxxxxxx...
True
False
```

Префикс `$2b$12$` — сигнатура bcrypt.

### Коммит

```powershell
git add backend/utils/security.py
git commit -m "feat(week-4): add password hashing and JWT"
```

---

## 4.3. `backend/api/deps.py` — зависимости FastAPI

**Что это:** файл с зависимостями (`Depends`) для FastAPI. Здесь живут функции:

- выдают сессию БД в роуты;
- проверяют JWT и возвращают текущего пользователя;
- проверяют роль пользователя (admin, editor).

**Что такое `Depends`:** механизм FastAPI для внедрения зависимостей. Если в сигнатуре роута написать `user: User = Depends(get_current_user)`, FastAPI вызовет `get_current_user` перед роутом и передаст результат в параметр `user`. Если `get_current_user` бросит `HTTPException`, роут не выполнится.

### Создание файла

Создайте `backend/api/deps.py`:

```python
# backend/api/deps.py
"""
Зависимости FastAPI: БД, текущий пользователь, проверка ролей.

Использование в роутах:

    @router.get("/protected")
    async def protected(user: User = Depends(get_current_user)):
        return {"email": user.email}

    @router.post("/admin-only")
    async def admin_only(user: User = Depends(require_admin)):
        return {"status": "ok"}
"""

import logging
from typing import Optional

from fastapi import Depends, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from sqlalchemy import select
from sqlalchemy.ext.asyncio import AsyncSession

from backend.db.models import User
from backend.db.session import AsyncSessionLocal
from backend.utils.security import decode_token

logger = logging.getLogger(__name__)


# ============================================================================
# Аутентификация через Bearer-токен
# ============================================================================
# auto_error=False — если заголовка Authorization нет, не бросать 403 сразу,
# а вернуть None. Так мы сможем использовать этот же объект
# и в обязательной (get_current_user), и в опциональной (optional_user) зависимости.
security = HTTPBearer(auto_error=False)


# ============================================================================
# Сессия БД
# ============================================================================
async def get_db() -> AsyncSession:
    """
    FastAPI-зависимость: выдаёт сессию БД и закрывает после ответа.

    ВАЖНО: этот вариант используется в защищённых роутах, где нужен
    доступ к БД. В простых случаях можно использовать get_async_session
    из backend.db.session — они эквивалентны.
    """
    async with AsyncSessionLocal() as session:
        yield session


# ============================================================================
# Текущий пользователь (обязательный)
# ============================================================================
async def get_current_user(
    credentials: Optional[HTTPAuthorizationCredentials] = Depends(security),
    db: AsyncSession = Depends(get_db),
) -> User:
    """
    Возвращает текущего пользователя по JWT-токену из заголовка Authorization.

    Бросает:
    - 401 — нет заголовка, токен невалидный или истёк.
    - 404 — токен валидный, но пользователь удалён или деактивирован.
    """
    if not credentials:
        raise HTTPException(status_code=401, detail="Not authenticated")

    token = credentials.credentials
    payload = decode_token(token)

    user_id = payload.get("sub")
    if not user_id:
        raise HTTPException(status_code=401, detail="Invalid token")

    result = await db.execute(
        select(User).where(User.id == user_id, User.is_active == True)  # noqa: E712
    )
    user = result.scalar_one_or_none()

    if not user:
        # Токен валидный, но пользователя нет или он деактивирован.
        raise HTTPException(status_code=404, detail="User not found")

    return user


# ============================================================================
# Текущий пользователь (опциональный)
# ============================================================================
async def optional_user(
    credentials: Optional[HTTPAuthorizationCredentials] = Depends(security),
    db: AsyncSession = Depends(get_db),
) -> Optional[User]:
    """
    Возвращает пользователя или None.

    Отличие от get_current_user: НИКОГДА не бросает исключений.
    Используется в публичных роутах, где хочется знать, залогинен ли
    пользователь, но не блокировать анонимный доступ.
    """
    if not credentials:
        return None

    try:
        payload = decode_token(credentials.credentials)
        user_id = payload.get("sub")
        if not user_id:
            return None

        result = await db.execute(
            select(User).where(User.id == user_id, User.is_active == True)  # noqa: E712
        )
        return result.scalar_one_or_none()
    except Exception:
        return None


# ============================================================================
# Проверка ролей
# ============================================================================
async def require_admin(current_user: User = Depends(get_current_user)) -> User:
    """Пропускает только администраторов. Иначе — 403."""
    if current_user.role != "admin":
        raise HTTPException(status_code=403, detail="Admin access required")
    return current_user


async def require_editor(current_user: User = Depends(get_current_user)) -> User:
    """Пропускает admin и editor. Иначе — 403."""
    if current_user.role not in ["admin", "editor"]:
        raise HTTPException(status_code=403, detail="Editor access required")
    return current_user
```

### Разбор ключевых решений

**Почему `HTTPBearer(auto_error=False)`:** если поставить `True`, FastAPI сам бросит 403 при отсутствии заголовка. Нам это не подходит — для анонимных запросов нужен 401, а для `optional_user` — вообще `None`.

**Почему `get_current_user` возвращает 404, а не 401, если юзера нет:** токен валидный, просто пользователя удалили или деактивировали. Это не «не аутентифицирован», а «ресурс не найден».

**Почему `User.is_active == True` с `# noqa: E712`:** SQLAlchemy формально предпочитает `.is_(True)`, чтобы избежать сравнения с `None`. Но `.is_(True)` не работает одинаково хорошо для SQLite и Postgres. `== True` работает везде. Мы используем `== True` и подавляем предупреждение линтера.

**Почему `optional_user` оборачивает всё в `try/except Exception`:** если токен невалидный, `decode_token` бросит `HTTPException(401)`. Публичный роут должен вернуть данные без пользователя. Ловим всё и возвращаем `None`.

**Почему `require_admin` использует `Depends(get_current_user)`, а не свой код:** не дублируем логику. Сначала `get_current_user` находит пользователя (или бросает 401/404), потом `require_admin` проверяет роль (или бросает 403).

### Проверка: импорты работают

```powershell
python -m poetry run python -c "from backend.api.deps import get_current_user, require_admin, require_editor, optional_user, security; print('OK')"
```

**Что вы должны увидеть:** `OK`.

### Коммит

```powershell
git add backend/api/deps.py
git commit -m "feat(week-4): add FastAPI dependencies (auth, roles, db)"
```

---

## 4.4. `backend/api/routes/auth.py` — роуты регистрации и логина

**Что делаем:** три роута:

- `POST /auth/register` — создаёт пользователя, возвращает JWT;
- `POST /auth/login` — проверяет email+пароль, возвращает JWT;
- `GET /auth/me` — возвращает данные текущего пользователя.

### Создание файла

Создайте `backend/api/routes/auth.py`:

```python
# backend/api/routes/auth.py
"""
Роуты аутентификации:

- POST /auth/register — регистрация, возвращает JWT.
- POST /auth/login    — вход, возвращает JWT.
- GET  /auth/me       — текущий пользователь (нужен JWT).
"""

import logging
import uuid

from fastapi import APIRouter, Depends, HTTPException, status
from pydantic import BaseModel, EmailStr, field_validator
from sqlalchemy import select
from sqlalchemy.ext.asyncio import AsyncSession

from backend.api.deps import get_current_user
from backend.db.models import User
from backend.db.session import get_async_session
from backend.utils.security import (
    create_access_token,
    get_password_hash,
    verify_password,
)

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/auth", tags=["auth"])


# ============================================================================
# Pydantic-модели запросов и ответов
# ============================================================================
class RegisterRequest(BaseModel):
    """Тело запроса на регистрацию."""

    email: EmailStr
    username: str
    password: str
    full_name: str | None = None

    @field_validator("password")
    def validate_password(cls, v: str) -> str:
        """Проверяет сложность пароля."""
        if len(v) < 8:
            raise ValueError("Password must be at least 8 characters")
        if not any(c.isdigit() for c in v):
            raise ValueError("Password must contain at least one digit")
        if not any(c.isupper() for c in v):
            raise ValueError("Password must contain at least one uppercase letter")
        return v


class LoginRequest(BaseModel):
    """Тело запроса на вход."""

    email: EmailStr
    password: str


class TokenResponse(BaseModel):
    """Ответ с JWT-токеном и данными пользователя."""

    access_token: str
    token_type: str = "bearer"
    user_id: str
    email: str
    role: str
    username: str


# ============================================================================
# POST /auth/register
# ============================================================================
@router.post(
    "/register",
    response_model=TokenResponse,
    status_code=status.HTTP_201_CREATED,
)
async def register(
    request: RegisterRequest,
    session: AsyncSession = Depends(get_async_session),
):
    """
    Регистрирует нового пользователя.

    Проверяет уникальность email и username, хеширует пароль,
    сохраняет пользователя и возвращает JWT-токен.

    Коды ответов:
    - 201 — успех.
    - 400 — email или username уже заняты.
    - 422 — невалидные данные (пароль слабый, email не email).
    """
    # Проверка email
    existing = await session.execute(
        select(User).where(User.email == request.email)
    )
    if existing.scalar_one_or_none():
        raise HTTPException(status_code=400, detail="Email already registered")

    # Проверка username
    existing = await session.execute(
        select(User).where(User.username == request.username)
    )
    if existing.scalar_one_or_none():
        raise HTTPException(status_code=400, detail="Username already taken")

    # Создание пользователя
    user = User(
        id=str(uuid.uuid4()),
        email=request.email,
        username=request.username,
        hashed_password=get_password_hash(request.password),
        full_name=request.full_name,
        role="user",
    )
    session.add(user)
    await session.commit()
    await session.refresh(user)

    # JWT
    access_token = create_access_token(data={"sub": user.id, "role": user.role})

    logger.info("User registered: %s", user.email)

    return TokenResponse(
        access_token=access_token,
        user_id=user.id,
        email=user.email,
        role=user.role,
        username=user.username,
    )


# ============================================================================
# POST /auth/login
# ============================================================================
@router.post("/login", response_model=TokenResponse)
async def login(
    request: LoginRequest,
    session: AsyncSession = Depends(get_async_session),
):
    """
    Выполняет вход.

    Коды ответов:
    - 200 — успех, возвращает токен.
    - 401 — неверный email или пароль.
    - 403 — аккаунт деактивирован.
    """
    user = await session.execute(select(User).where(User.email == request.email))
    user = user.scalar_one_or_none()

    # Одинаковый текст ошибки для «нет пользователя» и «неверный пароль»
    # — чтобы злоумышленник не мог проверить существование email перебором.
    if not user:
        raise HTTPException(status_code=401, detail="Invalid credentials")

    if not verify_password(request.password, user.hashed_password):
        raise HTTPException(status_code=401, detail="Invalid credentials")

    if not user.is_active:
        raise HTTPException(status_code=403, detail="Account is disabled")

    access_token = create_access_token(data={"sub": user.id, "role": user.role})

    logger.info("User logged in: %s", user.email)

    return TokenResponse(
        access_token=access_token,
        user_id=user.id,
        email=user.email,
        role=user.role,
        username=user.username,
    )


# ============================================================================
# GET /auth/me
# ============================================================================
@router.get("/me")
async def get_me(current_user: User = Depends(get_current_user)):
    """
    Возвращает данные текущего пользователя.

    Требует заголовок Authorization: Bearer <token>.
    """
    return {
        "id": current_user.id,
        "email": current_user.email,
        "username": current_user.username,
        "full_name": current_user.full_name,
        "role": current_user.role,
        "is_active": current_user.is_active,
    }
```

### Разбор ключевых решений

**Почему `EmailStr`:** Pydantic автоматически валидирует email по правилам. Требует пакет `email-validator` — он в зависимостях. Невалидный email → 422.

**Почему `field_validator("password")` без `@classmethod`:** в Pydantic v2 декоратор `@field_validator` уже оборачивает метод в classmethod автоматически. Явный `@classmethod` не нужен (но и не вредит).

**Почему один и тот же текст «Invalid credentials» при «нет пользователя» и «неверный пароль»:** не даём злоумышленнику возможности перебором email проверять, какие зарегистрированы.

**Почему 403 для деактивированного, а не 401:** 401 — «неправильные данные для входа». 403 — «данные верные, но доступ запрещён». Аккаунт `is_active=False` — это 403.

**Почему `await session.refresh(user)`:** после `commit()` объект может «протухнуть». `refresh` загружает актуальные поля (`created_at`, который сгенерировала БД через `server_default=func.now()`).

**Почему `role="user"` прописано явно:** в модели `role` имеет `default="user"`, но лучше не полагаться на неявные дефолты в бизнес-логике.

### Обновление `backend/api/main.py` — подключение роутера `auth`

Откройте `backend/api/main.py`. **Измените** импорт и список роутеров:

**Было (после Этапа 3):**

```python
from backend.api.routes import health
...
app.include_router(health.router)
```

**Стало:**

```python
from backend.api.routes import auth, health
...
app.include_router(auth.router)
app.include_router(health.router)
```

**Проверьте, что в файле нет импорта `translate`, `admin`, `eval`, `hitl`** — эти роутеры появятся на этапах 9–10. Если они есть в импорте, но файлов нет — приложение упадёт с `ModuleNotFoundError`.

**Полный актуальный `main.py` после Этапа 4:**

```python
# backend/api/main.py
"""
Точка входа FastAPI.

На этом этапе подключены роутеры auth и health.
Остальные роутеры (translate, admin, eval, hitl) появятся на этапах 9-10.
"""

from contextlib import asynccontextmanager

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from backend.api.routes import auth, health
from backend.core.config import get_settings
from backend.core.constants import LANGUAGES
from backend.db.models import Base
from backend.db.session import async_engine

settings = get_settings()


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Создание таблиц при старте
    async with async_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    yield


app = FastAPI(
    title=settings.PROJECT_NAME,
    debug=settings.DEBUG,
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"] if settings.DEBUG else [],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Роутеры на этом этапе
app.include_router(auth.router)
app.include_router(health.router)


@app.get("/")
async def root():
    return {"message": f"{settings.PROJECT_NAME} is running!"}


@app.get("/healthz")
async def healthz():
    return {"status": "ok"}


@app.get("/languages")
async def languages():
    return {"languages": LANGUAGES}
```

### Проверка: приложение запускается

```powershell
python -m poetry run python run.py
```

Откройте [http://localhost:8000/docs](http://localhost:8000/docs).

**Что вы должны увидеть:** секции **auth** (3 эндпоинта) и **health**. Секций `translate`, `admin`, `eval`, `hitl` нет — они появятся позже.

Остановите приложение (**Ctrl + C**).

### Коммит

```powershell
git add backend/api/routes/auth.py backend/api/main.py
git commit -m "feat(week-4): add auth routes (register, login, me)"
```

---

## 4.5. Ручная проверка через `curl` (или Swagger)

**Запустите приложение:**

```powershell
python -m poetry run python run.py
```

Откройте **новое** окно PowerShell.

### Регистрация

```powershell
curl -X POST "http://localhost:8000/auth/register" -H "Content-Type: application/json" -d "{\"email\":\"test@example.com\",\"username\":\"testuser\",\"password\":\"StrongPass1\",\"full_name\":\"Test User\"}"
```

**Что вы должны увидеть:**

```json
{"access_token":"eyJhbGciOiJIUzI1NiIs...","token_type":"bearer","user_id":"...","email":"test@example.com","role":"user","username":"testuser"}
```

Скопируйте значение `access_token` — оно понадобится.

### Логин

```powershell
curl -X POST "http://localhost:8000/auth/login" -H "Content-Type: application/json" -d "{\"email\":\"test@example.com\",\"password\":\"StrongPass1\"}"
```

**Что вы должны увидеть:** снова JSON с токеном.

### Получение текущего пользователя

Подставьте свой токен вместо `<TOKEN>`:

```powershell
curl -X GET "http://localhost:8000/auth/me" -H "Authorization: Bearer <TOKEN>"
```

**Что вы должны увидеть:**

```json
{"id":"...","email":"test@example.com","username":"testuser","full_name":"Test User","role":"user","is_active":true}
```

### Без токена

```powershell
curl -X GET "http://localhost:8000/auth/me"
```

**Что вы должны увидеть:** 401:

```json
{"detail":"Not authenticated"}
```

### Неверный пароль

```powershell
curl -X POST "http://localhost:8000/auth/login" -H "Content-Type: application/json" -d "{\"email\":\"test@example.com\",\"password\":\"WrongPass1\"}"
```

**Что вы должны увидеть:** 401 `{"detail":"Invalid credentials"}`.

### Слабый пароль

```powershell
curl -X POST "http://localhost:8000/auth/register" -H "Content-Type: application/json" -d "{\"email\":\"weak@example.com\",\"username\":\"weak\",\"password\":\"weak\"}"
```

**Что вы должны увидеть:** 422 с деталями валидации.

### Повторная регистрация с тем же email

```powershell
curl -X POST "http://localhost:8000/auth/register" -H "Content-Type: application/json" -d "{\"email\":\"test@example.com\",\"username\":\"other\",\"password\":\"StrongPass1\"}"
```

**Что вы должны увидеть:** 400 `{"detail":"Email already registered"}`.

### Проверка, что в БД хранится хеш

Если Docker запущен, во **втором** окне из папки `infrastructure`:

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT email, LEFT(hashed_password, 7) AS prefix FROM users;"
```

**Что вы должны увидеть:**

```
      email       | prefix
------------------+---------
 test@example.com | $2b$12$
```

Префикс `$2b$12$` — bcrypt. **Пароль в открытом виде нигде не хранится.**

Если вы работаете без Docker (с SQLite):

```powershell
python -m poetry run python -c "import sqlite3; conn = sqlite3.connect('rung.db'); cur = conn.cursor(); cur.execute('SELECT email, LEFT(hashed_password, 7) FROM users'); print(cur.fetchall())"
```

**Остановите приложение** (Ctrl + C).

---

## 4.6. `tests/unit/test_security.py` — тесты bcrypt и JWT

**Что тестируем:**

1. `get_password_hash` возвращает bcrypt-хеш (`$2b$`).
2. Хеш одного пароля разный при повторных вызовах (соль работает).
3. `verify_password` возвращает `True` для правильного пароля.
4. `verify_password` возвращает `False` для неправильного.
5. `create_access_token` создаёт токен с тремя частями.
6. `decode_token` возвращает payload с `sub` и `exp`.
7. Истёкший токен → `HTTPException(401)`.
8. Токен с неверной подписью → `HTTPException(401)`.
9. Malformed-токен → `HTTPException(401)`.

### Создание файла

Создайте `tests/unit/test_security.py`:

```python
# tests/unit/test_security.py
"""
Юнит-тесты для backend.utils.security.
"""

from datetime import timedelta

import pytest
from fastapi import HTTPException
from jose import jwt

from backend.utils.security import (
    ALGORITHM,
    create_access_token,
    decode_token,
    get_password_hash,
    verify_password,
)


# ============================================================================
# Хеширование паролей
# ============================================================================
class TestPasswordHashing:
    def test_hash_starts_with_bcrypt_prefix(self):
        h = get_password_hash("StrongPass1")
        assert h.startswith("$2b$") or h.startswith("$2a$")

    def test_same_password_different_hashes(self):
        """bcrypt добавляет соль → хеши разные."""
        h1 = get_password_hash("StrongPass1")
        h2 = get_password_hash("StrongPass1")
        assert h1 != h2

    def test_verify_correct_password(self):
        h = get_password_hash("StrongPass1")
        assert verify_password("StrongPass1", h) is True

    def test_verify_wrong_password(self):
        h = get_password_hash("StrongPass1")
        assert verify_password("WrongPass1", h) is False

    def test_verify_empty_password(self):
        h = get_password_hash("StrongPass1")
        assert verify_password("", h) is False

    def test_hash_is_60_chars(self):
        """Стандартный bcrypt-хеш — 60 символов."""
        h = get_password_hash("StrongPass1")
        assert len(h) == 60


# ============================================================================
# JWT-токены
# ============================================================================
class TestJWTCreation:
    def test_token_has_three_parts(self):
        token = create_access_token({"sub": "user-1", "role": "user"})
        parts = token.split(".")
        assert len(parts) == 3

    def test_decode_returns_payload(self):
        token = create_access_token({"sub": "user-1", "role": "admin"})
        payload = decode_token(token)
        assert payload["sub"] == "user-1"
        assert payload["role"] == "admin"
        assert "exp" in payload

    def test_custom_expires_delta(self):
        """Можно передать своё время жизни (для тестов)."""
        token = create_access_token(
            {"sub": "user-1"},
            expires_delta=timedelta(minutes=5),
        )
        payload = decode_token(token)
        assert payload["sub"] == "user-1"

    def test_expired_token_raises_401(self):
        """Токен с истёкшим exp → 401."""
        token = create_access_token(
            {"sub": "user-1"},
            expires_delta=timedelta(seconds=-1),
        )
        with pytest.raises(HTTPException) as exc_info:
            decode_token(token)
        assert exc_info.value.status_code == 401
        assert exc_info.value.detail == "Token expired"

    def test_invalid_signature_raises_401(self):
        """Токен, подписанный другим ключом, не пройдёт проверку."""
        payload = {"sub": "user-1", "exp": 9999999999}
        wrong_token = jwt.encode(payload, "wrong-secret-key", algorithm=ALGORITHM)
        with pytest.raises(HTTPException) as exc_info:
            decode_token(wrong_token)
        assert exc_info.value.status_code == 401
        assert exc_info.value.detail == "Invalid token"

    def test_malformed_token_raises_401(self):
        """Строка, не похожая на JWT."""
        with pytest.raises(HTTPException) as exc_info:
            decode_token("not-a-jwt")
        assert exc_info.value.status_code == 401

    def test_token_sub_string(self):
        """Если передали не-строку в sub, она становится строкой."""
        token = create_access_token({"sub": 12345})
        payload = decode_token(token)
        assert payload["sub"] == "12345"
```

### Разбор

**Почему проверяем `$2b$`:** это сигнатура bcrypt. Если бы passlib использовал другой алгоритм — тест бы упал.

**Почему `test_same_password_different_hashes`:** проверка соли. Если хеши одинаковые — bcrypt не используется или без соли.

**Почему `timedelta(seconds=-1)`:** создаёт уже истёкший токен. `jose` поймает и бросит `ExpiredSignatureError`.

**Почему `jwt.encode(payload, "wrong-secret-key", ...)`:** создаём токен с другим ключом. Наш `decode_token` использует правильный ключ — подпись не сойдётся.

**Почему `test_hash_is_60_chars`:** bcrypt всегда даёт 60 символов. Если меньше — что-то не так.

### Запуск

```powershell
python -m poetry run pytest tests/unit/test_security.py -v
```

**Что вы должны увидеть:**

```
tests/unit/test_security.py::TestPasswordHashing::test_hash_starts_with_bcrypt_prefix PASSED
...
============================= 12 passed in 0.8s ==============================
```

### Коммит

```powershell
git add tests/unit/test_security.py
git commit -m "test(week-4): add unit tests for security"
```

---

## 4.7. `tests/integration/test_auth.py` — интеграционные тесты

**Что тестируем:** полный цикл register → login → `/me` через `TestClient` с SQLite in-memory.

**Ключевая сложность:** приложение создаёт `async_engine` при импорте `backend.db.session` с URL из `.env`. Нам нужно подменить его на in-memory SQLite.

**Как подменить правильно:**

- `backend.db.session.async_engine` — для lifespan и health;
- `backend.db.session.AsyncSessionLocal` — для `get_async_session` (в `session.py`);
- `backend.api.deps.AsyncSessionLocal` — для `get_db` (в `deps.py`, потому что там **прямой импорт**).

**Важно:** функция `get_async_session` использует глобальное имя `AsyncSessionLocal` в **своём модуле** (`backend.db.session`). А функция `get_db` в `deps.py` использует глобальное имя `AsyncSessionLocal` в **своём модуле** (`backend.api.deps`), потому что импортирует его напрямую. Поэтому патчить надо **в обоих модулях**.

### Создание файла

Создайте `tests/integration/test_auth.py`:

```python
# tests/integration/test_auth.py
"""
Интеграционные тесты для /auth/*.

Подменяем async_engine и AsyncSessionLocal на SQLite in-memory,
чтобы тесты не требовали ни Postgres, ни Docker.
"""

import asyncio

import pytest
from fastapi.testclient import TestClient
from sqlalchemy.ext.asyncio import (
    AsyncSession,
    async_sessionmaker,
    create_async_engine,
)
from sqlalchemy.pool import StaticPool

from backend.db.models import Base


# ============================================================================
# Фикстура: тестовое приложение с in-memory SQLite
# ============================================================================
@pytest.fixture
def client(monkeypatch):
    """
    Поднимает TestClient с изолированной SQLite in-memory.

    Подменяем:
    - db.session.async_engine       — для lifespan и health
    - db.session.AsyncSessionLocal  — для get_async_session (в session.py)
    - api.deps.AsyncSessionLocal    — для get_db (в deps.py, там прямой импорт)
    """
    # StaticPool: одно общее соединение на всё время работы фикстуры.
    # Иначе in-memory SQLite создаёт новую БД на каждое соединение,
    # и таблицы «пропадают» между запросами.
    test_engine = create_async_engine(
        "sqlite+aiosqlite:///:memory:",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
    )
    test_session_local = async_sessionmaker(
        bind=test_engine,
        class_=AsyncSession,
        expire_on_commit=False,
    )

    # Подменяем объекты в модулях
    import backend.db.session as db_session_module
    import backend.api.deps as deps_module

    monkeypatch.setattr(db_session_module, "async_engine", test_engine)
    monkeypatch.setattr(db_session_module, "AsyncSessionLocal", test_session_local)
    monkeypatch.setattr(deps_module, "AsyncSessionLocal", test_session_local)

    # Создаём таблицы
    async def _create_tables():
        async with test_engine.begin() as conn:
            await conn.run_sync(Base.metadata.create_all)

    asyncio.run(_create_tables())

    # Импортируем app ПОСЛЕ подмены
    from backend.api.main import app

    with TestClient(app) as c:
        yield c


# ============================================================================
# Фикстура: зарегистрированный пользователь
# ============================================================================
@pytest.fixture
def registered_user(client):
    """Регистрирует пользователя и возвращает словарь с данными и токеном."""
    r = client.post("/auth/register", json={
        "email": "user@example.com",
        "username": "user1",
        "password": "StrongPass1",
        "full_name": "User One",
    })
    assert r.status_code == 201, r.text
    data = r.json()
    return {
        "email": "user@example.com",
        "username": "user1",
        "password": "StrongPass1",
        "token": data["access_token"],
        "user_id": data["user_id"],
    }


# ============================================================================
# Регистрация
# ============================================================================
class TestRegister:
    def test_register_success(self, client):
        r = client.post("/auth/register", json={
            "email": "new@example.com",
            "username": "newuser",
            "password": "StrongPass1",
            "full_name": "New User",
        })
        assert r.status_code == 201
        data = r.json()
        assert "access_token" in data
        assert data["email"] == "new@example.com"
        assert data["username"] == "newuser"
        assert data["role"] == "user"
        assert data["token_type"] == "bearer"

    def test_register_duplicate_email(self, client, registered_user):
        r = client.post("/auth/register", json={
            "email": registered_user["email"],
            "username": "different_username",
            "password": "StrongPass1",
        })
        assert r.status_code == 400
        assert r.json()["detail"] == "Email already registered"

    def test_register_duplicate_username(self, client, registered_user):
        r = client.post("/auth/register", json={
            "email": "different@example.com",
            "username": registered_user["username"],
            "password": "StrongPass1",
        })
        assert r.status_code == 400
        assert r.json()["detail"] == "Username already taken"

    def test_register_weak_password_too_short(self, client):
        r = client.post("/auth/register", json={
            "email": "weak@example.com",
            "username": "weakuser",
            "password": "Short1",
        })
        assert r.status_code == 422
        assert "at least 8 characters" in r.text

    def test_register_weak_password_no_digit(self, client):
        r = client.post("/auth/register", json={
            "email": "weak@example.com",
            "username": "weakuser",
            "password": "NoDigitsHere",
        })
        assert r.status_code == 422
        assert "digit" in r.text

    def test_register_weak_password_no_uppercase(self, client):
        r = client.post("/auth/register", json={
            "email": "weak@example.com",
            "username": "weakuser",
            "password": "nouppercase1",
        })
        assert r.status_code == 422
        assert "uppercase" in r.text

    def test_register_invalid_email(self, client):
        r = client.post("/auth/register", json={
            "email": "not-an-email",
            "username": "someuser",
            "password": "StrongPass1",
        })
        assert r.status_code == 422


# ============================================================================
# Логин
# ============================================================================
class TestLogin:
    def test_login_success(self, client, registered_user):
        r = client.post("/auth/login", json={
            "email": registered_user["email"],
            "password": registered_user["password"],
        })
        assert r.status_code == 200
        data = r.json()
        assert "access_token" in data
        assert data["email"] == registered_user["email"]

    def test_login_wrong_password(self, client, registered_user):
        r = client.post("/auth/login", json={
            "email": registered_user["email"],
            "password": "WrongPass1",
        })
        assert r.status_code == 401
        assert r.json()["detail"] == "Invalid credentials"

    def test_login_unknown_email(self, client):
        r = client.post("/auth/login", json={
            "email": "nobody@example.com",
            "password": "StrongPass1",
        })
        assert r.status_code == 401
        assert r.json()["detail"] == "Invalid credentials"


# ============================================================================
# /auth/me
# ============================================================================
class TestMe:
    def test_me_with_valid_token(self, client, registered_user):
        r = client.get(
            "/auth/me",
            headers={"Authorization": f"Bearer {registered_user['token']}"},
        )
        assert r.status_code == 200
        data = r.json()
        assert data["email"] == registered_user["email"]
        assert data["username"] == registered_user["username"]
        assert data["role"] == "user"
        assert data["is_active"] is True

    def test_me_without_token(self, client):
        r = client.get("/auth/me")
        assert r.status_code == 401

    def test_me_with_invalid_token(self, client):
        r = client.get(
            "/auth/me",
            headers={"Authorization": "Bearer not-a-valid-token"},
        )
        assert r.status_code == 401

    def test_me_after_login_token(self, client, registered_user):
        """Получаем новый токен через логин и используем его."""
        r = client.post("/auth/login", json={
            "email": registered_user["email"],
            "password": registered_user["password"],
        })
        token = r.json()["access_token"]

        r = client.get("/auth/me", headers={"Authorization": f"Bearer {token}"})
        assert r.status_code == 200
        assert r.json()["email"] == registered_user["email"]


# ============================================================================
# Полный цикл
# ============================================================================
class TestFullCycle:
    def test_register_login_me(self, client):
        """Регистрация → /me → логин → /me — всё за один тест."""
        # Регистрация
        r = client.post("/auth/register", json={
            "email": "cycle@example.com",
            "username": "cycleuser",
            "password": "StrongPass1",
        })
        assert r.status_code == 201
        token_register = r.json()["access_token"]

        # /me с токеном регистрации
        r = client.get(
            "/auth/me",
            headers={"Authorization": f"Bearer {token_register}"},
        )
        assert r.status_code == 200
        assert r.json()["email"] == "cycle@example.com"

        # Логин
        r = client.post("/auth/login", json={
            "email": "cycle@example.com",
            "password": "StrongPass1",
        })
        assert r.status_code == 200
        token_login = r.json()["access_token"]

        # /me с токеном логина
        r = client.get(
            "/auth/me",
            headers={"Authorization": f"Bearer {token_login}"},
        )
        assert r.status_code == 200
        assert r.json()["email"] == "cycle@example.com"
```

### Разбор ключевых решений

**Почему `poolclass=StaticPool`:** для in-memory SQLite соединение живёт только пока открыт хотя бы один клиент. Если SQLAlchemy закроет соединение — таблицы исчезнут. `StaticPool` держит одно общее соединение, таблицы сохраняются.

**Почему патчим `deps.AsyncSessionLocal`:** `deps.py` делает `from backend.db.session import AsyncSessionLocal` — это **прямой импорт**, значит в модуле `deps` есть своё собственное имя `AsyncSessionLocal`, которое не изменится при патче `db.session.AsyncSessionLocal`. Нужно патчить в обоих местах.

**Почему `monkeypatch` вместо прямого присваивания:** `monkeypatch.setattr` автоматически откатывает изменения после теста. Так тесты не влияют друг на друга.

**Почему `asyncio.run(_create_tables())`:** создание таблиц — async-операция, а pytest-фикстура синхронная. Запускаем один раз на все тесты.

**Почему `with TestClient(app) as c`:** контекст-менеджер запускает lifespan приложения (создание таблиц), а потом корректно его останавливает. Если использовать `TestClient(app)` без `with`, lifespan не выполнится.

**Почему `test_me_with_invalid_token` ожидает 401, а не 500:** наш `decode_token` ловит `JWTError` и бросает `HTTPException(401)`. `get_current_user` пробрасывает его дальше.

### Запуск

```powershell
python -m poetry run pytest tests/integration/test_auth.py -v
```

**Что вы должны увидеть:**

```
tests/integration/test_auth.py::TestRegister::test_register_success PASSED
tests/integration/test_auth.py::TestRegister::test_register_duplicate_email PASSED
...
tests/integration/test_auth.py::TestFullCycle::test_register_login_me PASSED
============================= 16 passed in 2.3s ==============================
```

**Если падает с `ModuleNotFoundError`:** убедитесь, что в `pyproject.toml` есть `pythonpath = ["."]`.

**Если падает с `sqlite3.OperationalError: no such table`:** проверьте, что используется `poolclass=StaticPool` и что `asyncio.run(_create_tables())` вызывается **до** создания `TestClient`.

**Если `test_me_with_valid_token` возвращает 500:** вероятно, вы забыли пропатчить `deps_module.AsyncSessionLocal`. `get_current_user` использует `get_db`, а тот — свою копию `AsyncSessionLocal`.

### Коммит

```powershell
git add tests/integration/test_auth.py
git commit -m "test(week-4): add integration tests for auth"
```

---

## 4.8. Финальная проверка этапа

### 1. Все тесты проходят

```powershell
python -m poetry run pytest tests/ -v
```

**Что вы должны увидеть:**

- `test_config.py` — 17 PASSED
- `test_health.py` — 8 PASSED
- `test_models.py` — 11 PASSED
- `test_security.py` — 12 PASSED
- `test_auth.py` — 16 PASSED

Итого: ~64 теста, все зелёные.

### 2. Приложение запускается

```powershell
python -m poetry run python run.py
```

Открывается `/docs`, видны секции **auth** и **health**. Секций `translate`, `admin`, `eval`, `hitl` нет — это правильно.

### 3. Ручной цикл

- `POST /auth/register` → 201 с токеном.
- `POST /auth/login` → 200 с токеном.
- `GET /auth/me` с токеном → 200 с данными.
- `GET /auth/me` без токена → 401.
- `POST /auth/register` с тем же email → 400.
- `POST /auth/register` со слабым паролем → 422.

### 4. Хеш в БД

```powershell
cd infrastructure
docker compose exec db psql -U rung_user -d rung_db -c "SELECT email, LEFT(hashed_password, 7) FROM users;"
```

**Что вы должны увидеть:** `$2b$12$` — bcrypt.

### 5. Слияние

```powershell
cd ..
git add .
git commit -m "chore(week-4): finalize week 4"   # если есть что коммитить
git checkout main
git merge week-4 --no-ff -m "merge: week-4 (authentication)"
git push origin main
git branch -d week-4
```

---

## 4.9. CRUD для таблицы `users` через `docker compose exec`

На этом этапе **админ-роутов ещё нет** (они появятся на этапе 9). Все операции с пользователями делаем через `psql` в контейнере `db` или через скрипт `scripts/create_admin.py`.

**Все команды — из папки `infrastructure`.** Если из корня — используйте `docker compose -f infrastructure/docker-compose.yml exec ...`.

### CREATE — создание пользователей

**Через API (рекомендуемый способ):**

```powershell
curl -X POST "http://localhost:8000/auth/register" -H "Content-Type: application/json" -d "{\"email\":\"admin@example.com\",\"username\":\"admin\",\"password\":\"StrongPass1\"}"
```

**Через готовый скрипт (для админов):**

```powershell
python -m poetry run python scripts/create_admin.py --email admin@example.com --username admin --password AdminPass1
```

Скрипт сам хеширует пароль через `get_password_hash` и вставляет пользователя с ролью `admin`. Самый удобный способ.

**Через SQL (если нужно вставить с заранее известным хешем):**

```powershell
$hash = python -m poetry run python -c "from backend.utils.security import get_password_hash; print(get_password_hash('AdminPass1'))"
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO users (id, email, username, hashed_password, full_name, role, is_active) VALUES ('admin-001', 'admin@example.com', 'admin', '$hash', 'Administrator', 'admin', true) ON CONFLICT (email) DO NOTHING;"
```

**Важно:** знаки `$` в PowerShell нужно экранировать, но через переменную `$hash` всё подставляется автоматически.

### READ — чтение пользователей

**Все пользователи:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, email, username, role, is_active FROM users;"
```

**Конкретный пользователь по email:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT id, email, username, role, is_active FROM users WHERE email = 'admin@example.com';"
```

**Проверка, что пароль захеширован:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT email, LEFT(hashed_password, 7) AS prefix, LENGTH(hashed_password) AS length FROM users;"
```

**Что вы должны увидеть:** `prefix = $2b$12$`, `length = 60`.

**Пользователи с ролью admin:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT email, username FROM users WHERE role = 'admin';"
```

**Активные пользователи:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT email, role FROM users WHERE is_active = true;"
```

**Статистика по ролям:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT role, COUNT(*) FROM users GROUP BY role;"
```

**Структура таблицы (метакоманда):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "\d users"
```

### UPDATE — изменение пользователей

**Сменить роль на admin:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET role = 'admin' WHERE email = 'user@example.com';"
```

**Сменить роль на editor:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET role = 'editor' WHERE email = 'user@example.com';"
```

**Деактивировать пользователя:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET is_active = false WHERE email = 'spam@example.com';"
```

**Сменить пароль (сгенерировать новый хеш):**

```powershell
$hash = python -m poetry run python -c "from backend.utils.security import get_password_hash; print(get_password_hash('NewPass123'))"
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET hashed_password = '$hash' WHERE email = 'user@example.com';"
```

**Проверить, что новый пароль работает:**

```powershell
curl -X POST "http://localhost:8000/auth/login" -H "Content-Type: application/json" -d "{\"email\":\"user@example.com\",\"password\":\"NewPass123\"}"
```

**Что вы должны увидеть:** 200 и токен.

**Обновить `full_name`:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET full_name = 'Real Name' WHERE email = 'user@example.com';"
```

### DELETE — удаление пользователей

**Удалить по email:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM users WHERE email = 'spam@example.com';"
```

**Удалить по id:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM users WHERE id = 'u-001';"
```

**Удалить всех, кроме админов:**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "DELETE FROM users WHERE role != 'admin';"
```

**Очистить всех (осторожно):**

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "TRUNCATE TABLE users CASCADE;"
```

### Комбинированные сценарии

**Создать админа через SQL — одной командой:**

```powershell
$hash = python -m poetry run python -c "from backend.utils.security import get_password_hash; print(get_password_hash('AdminPass123'))"
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO users (id, email, username, hashed_password, role, is_active) VALUES ('admin-001', 'admin@rung.local', 'admin', '$hash', 'admin', true) ON CONFLICT (email) DO NOTHING;"
```

**Проверить полный цикл через `psql` и `curl`:**

```powershell
# 1. Создать пользователя через SQL
$hash = python -m poetry run python -c "from backend.utils.security import get_password_hash; print(get_password_hash('TestPass123'))"
docker compose exec db psql -U rung_user -d rung_db -c "INSERT INTO users (id, email, username, hashed_password, role, is_active) VALUES ('u-test', 'test@rung.local', 'testuser', '$hash', 'user', true);"

# 2. Залогиниться через API
curl -X POST "http://localhost:8000/auth/login" -H "Content-Type: application/json" -d "{\"email\":\"test@rung.local\",\"password\":\"TestPass123\"}"
```

**Что вы должны увидеть:** 200 и токен. Доказывает, что хеш, сгенерированный через `get_password_hash`, совместим с `verify_password`.

### Интерактивный режим

Для работы с множеством команд откройте интерактивную сессию:

```powershell
docker compose exec db psql -U rung_user -d rung_db
```

Внутри — обычный `psql`: `\dt`, `\d users`, `SELECT * FROM users;`, выход — `\q`.

---

## 4.10. Структура проекта после Этапа 4

```
RUNG/
├── .editorconfig
├── .env                          # локально, не в Git
├── .env.example
├── .gitattributes
├── .gitignore
├── README.md
├── poetry.lock
├── pyproject.toml
├── run.py
├── rung.db                       # локальная SQLite, не в Git
│
├── backend/
│   ├── __init__.py
│   ├── api/
│   │   ├── __init__.py
│   │   ├── deps.py                            ← новый (week-4)
│   │   ├── main.py                            ← обновлён (подключён auth.router)
│   │   └── routes/
│   │       ├── __init__.py
│   │       ├── auth.py                        ← новый (week-4)
│   │       └── health.py
│   ├── core/
│   │   ├── __init__.py
│   │   ├── config.py
│   │   ├── constants.py
│   │   └── exceptions.py
│   ├── db/
│   │   ├── __init__.py
│   │   ├── session.py
│   │   ├── models.py
│   │   └── database.py
│   ├── utils/
│   │   ├── __init__.py
│   │   ├── logging.py
│   │   └── security.py                        ← новый (week-4)
│   ├── agents/__init__.py                     # пусто (этап 5)
│   ├── data/__init__.py                       # пусто (этап 6)
│   ├── hitl/__init__.py                       # пусто (этап 10)
│   ├── retrieval/__init__.py                  # пусто (этап 7)
│   └── worker/__init__.py                     # пусто (этап 11)
│
├── data/
│   └── import/
│       └── .gitkeep
│
├── infrastructure/
│   ├── Dockerfile
│   ├── docker-compose.yml
│   ├── .dockerignore
│   ├── .env                                   # локально, не в Git
│   └── .env.example
│
├── logs/
│   ├── .gitkeep
│   ├── rung_YYYYMMDD.log
│   └── errors_YYYYMMDD.log
│
├── scripts/
│   └── create_admin.py                        # уже был, теперь применим
│
└── tests/
    ├── __init__.py
    ├── integration/
    │   ├── __init__.py
    │   ├── test_auth.py                       ← новый (week-4)
    │   └── test_health.py
    └── unit/
        ├── __init__.py
        ├── test_config.py
        ├── test_models.py
        └── test_security.py                   ← новый (week-4)
```

### Как проверить

```powershell
tree /F /A
```

Сверьте с приведённым деревом.

### Каких файлов ещё НЕ должно быть

- `backend/agents/*` (кроме `__init__.py`) — этап 5.
- `backend/data/*` (кроме `__init__.py`) — этапы 6, 7.
- `backend/retrieval/*` — этап 7.
- `backend/hitl/*` — этап 10.
- `backend/eval/*` — этап 9б.
- `backend/worker/celery_app.py`, `backend/worker/tasks.py` — этап 11.
- `backend/api/routes/translate.py`, `admin.py`, `hitl.py`, `eval.py` — этапы 9, 10.
- `backend/utils/llm_client.py`, `hf_llm.py`, `llm_processing.py`, `async_utils.py` — этап 8.
- `backend/core/models.py` — этап 9.
- `infrastructure/nginx.conf` — этап 12.
- `streamlit_app/` — этап 12.

**Если вы видите эти файлы у себя — значит, вы забежали вперёд.** Каждый файл появится на своём этапе.

---

## 4.11. Troubleshooting

### `AttributeError: module 'bcrypt' has no attribute '__about__'`

**Симптом:** при импорте `security.py` падает или пишет предупреждение.

**Причина:** установлен `bcrypt>=5.0`, а `passlib==1.7.4` не умеет с ним работать.

**Решение:**

```powershell
python -m poetry add "bcrypt<5.0.0"
python -m poetry install
```

Проверьте, что в `pyproject.toml` стоит `bcrypt = "<5.0.0"`.

### `jose.exceptions.JWTError: Signature verification failed`

**Симптом:** `decode_token` падает на токенах, которые вроде бы валидны.

**Причина 1:** `JWT_SECRET_KEY` в `.env` поменялся, а токен был выпущен со старым.

**Решение:** перевыпустить токен — заново залогиниться.

**Причина 2:** `JWT_ALGORITHM` в `.env` отличается от того, которым подписан токен.

**Решение:** проверьте `.env` — должно быть `HS256`.

### `sqlalchemy.exc.MissingGreenlet` в тестах

**Симптом:** тесты падают с этой ошибкой при попытке получить атрибут пользователя.

**Причина:** где-то используется `expire_on_commit=True`, или сессия закрывается раньше, чем читаются атрибуты.

**Решение:** убедитесь, что в `async_sessionmaker` стоит `expire_on_commit=False`.

### `RuntimeError: Event loop is closed` в `test_auth.py`

**Симптом:** тесты падают при попытке создать таблицы или сделать запрос.

**Причина:** `asyncio.run()` создал event loop, а потом закрыл его. Async engine привязан к этому loop'у.

**Решение:** используйте `TestClient(app)` как контекст-менеджер — тогда FastAPI сам управляет loop'ом. У нас так и сделано (`with TestClient(app) as c`).

### `sqlite3.OperationalError: no such table: users`

**Симптом:** тест регистрации возвращает 500, в логах — `no such table: users`.

**Причина:** таблицы создались в другом соединении, а запрос уходит в новое.

**Решение:** проверьте, что используется `poolclass=StaticPool`. Без него каждое соединение создаёт новую in-memory БД.

### `test_me_with_valid_token` возвращает 500, а не 200

**Симптом:** `/auth/me` с токеном падает.

**Причина:** вероятно, вы пропатчили `AsyncSessionLocal` только в `db.session`, но не в `deps`. `deps.py` делает прямой импорт `from backend.db.session import AsyncSessionLocal`, и его собственное имя остаётся ссылкой на оригинал.

**Решение:** добавьте `monkeypatch.setattr(deps_module, "AsyncSessionLocal", test_session_local)` в фикстуру.

### `403 Forbidden` при обращении к `/auth/me`

**Симптом:** с токеном получаете 403, а не 200.

**Причина:** скорее всего, не тот формат заголовка.

**Решение:** проверьте формат: `Authorization: Bearer eyJ...`. Между `Bearer` и токеном — **один пробел**.

### `psql` не находит таблицу `users`

**Симптом:** `docker compose exec db psql ...` пишет `relation "users" does not exist`.

**Причина:** таблицы создаются при старте приложения. Если `app` не стартовал — таблиц нет.

**Решение:** проверьте `docker compose ps` — `app` должен быть `healthy`. Посмотрите логи: `docker compose logs app`.

### Токен не истекает в тестах

**Симптом:** тест на истечение не падает, хотя должен.

**Причина:** `timedelta(seconds=-1)` — но jose может не проверять `exp`, если он в прошлом. Или системные часы смещены.

**Решение:** убедитесь, что токен создаётся с `expires_delta=timedelta(seconds=-1)` (не `+1`). Проверьте, что `datetime.now(timezone.utc)` — aware.

### `ImportError: cannot import name 'get_current_user' from backend.api.deps`

**Симптом:** импорт падает.

**Причина:** файл `deps.py` не создан или опечатка в имени функции.

**Решение:** проверьте, что файл есть (`dir backend\api\deps.py`) и функция называется точно `get_current_user`.

### `422 Unprocessable Entity` при валидных данных

**Симптом:** `POST /auth/register` с корректным паролем возвращает 422.

**Причина:** скорее всего, `EmailStr` не может валидировать email. Пакет `email-validator` не установлен.

**Решение:** `python -m poetry install` — или проверьте, что `email-validator` в `pyproject.toml`.

### `ModuleNotFoundError: No module named 'backend.api.routes.translate'`

**Симптом:** приложение не запускается после редактирования `main.py`.

**Причина:** в `main.py` остался импорт `from backend.api.routes import translate`. Этот файл появится только на этапе 9.

**Решение:** уберите лишние импорты. На этом этапе только `auth` и `health`.

---

## Итог этапа 4

К концу этапа у вас есть:

**Файлы кода:**

- `backend/utils/security.py` — bcrypt + JWT (хеширование, создание/декодирование токена);
- `backend/api/deps.py` — `get_db`, `get_current_user`, `require_admin`, `require_editor`, `optional_user`, `security = HTTPBearer`;
- `backend/api/routes/auth.py` — `/auth/register`, `/auth/login`, `/auth/me` + Pydantic-модели `RegisterRequest`, `LoginRequest`, `TokenResponse`;
- `backend/api/main.py` — обновлён, подключены только `auth.router` и `health.router`.

**Тесты:**

- `tests/unit/test_security.py` — 12 тестов на bcrypt и JWT;
- `tests/integration/test_auth.py` — 16 тестов на полный цикл аутентификации.

**Что работает:**

- Регистрация → возвращает JWT.
- Логин → возвращает JWT.
- `/auth/me` с токеном → данные пользователя.
- `/auth/me` без токена → 401.
- Пароли хранятся только как bcrypt-хеш (`$2b$12$...`, длина 60).
- Токен живёт `JWT_EXPIRATION_MINUTES` минут.
- Тесты проходят без реальной Postgres (SQLite in-memory).
- Пользователь может быть создан через `psql` в контейнере или через `scripts/create_admin.py`.

**Что дальше — Этап 5: Мультиагентная система (Agent, Validator, Critic, Editor, Supervisor)**

Это самый большой по количеству кода этап. Мы разобьём его на **две недели**:

**Первая неделя (5.1):**
- `backend/agents/base.py` — абстрактный класс `Agent`;
- `backend/agents/supervisor.py` — оркестратор с явным критерием выхода из цикла;
- `tests/unit/agents/test_supervisor.py` — тесты с mock-агентами.

**Вторая неделя (5.2):**
- `backend/agents/validator.py` — проверка по формальным правилам (без LLM);
- `backend/agents/critic.py` — LLM-as-a-Judge с парсингом JSON и fallback;
- `backend/agents/editor.py` — исправление с условным вызовом LLM;
- тесты на каждого агента.

Пока все агенты работают **с заглушками LLM** — реальная интеграция с Ollama и HuggingFace будет на этапе 8.